# FAFuse-Pro — ISIC 2018 Skin Lesion Analysis
Joint multitask model: segmentation guides classification via seg-guided mask.  
Architecture: ResNet-34 + DeiT-Small + ASG + CSAF + Deep Supervision  
**v2: Separate seg/cls training phases · FocalLoss · WeightedRandomSampler · MixUp · StepLR · ES**

In [1]:
pip install -q timm albumentations einops

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, glob, math, time, json, warnings

import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.amp import autocast, GradScaler

from torchvision import transforms
import timm

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, classification_report,
    precision_score, recall_score
)

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 14.6 GB


In [3]:
# ── Paths ────────────────────────────────────────────────
SEG_ROOT         = "/kaggle/input/datasets/tushargaur03/isic2018/isic2018/isic2018/Task1"
CLS_ROOT         = "/kaggle/input/datasets/tushargaur03/isic2018/isic2018/isic2018/Task3"
TEST_CSV_OVERRIDE= "/kaggle/input/datasets/tushargaur03/isic2018/ISIC2018_Task3_Test_GroundTruth.csv"
OUTPUT_DIR       = '/kaggle/working/fafuse_pro'
SEG_CKPT_DIR    = os.path.join(OUTPUT_DIR, 'seg_checkpoints')
CLS_CKPT_DIR    = os.path.join(OUTPUT_DIR, 'cls_checkpoints')
SEG_LOG_PATH    = os.path.join(OUTPUT_DIR, 'seg_log.json')
CLS_LOG_PATH    = os.path.join(OUTPUT_DIR, 'cls_log.json')
for d in [OUTPUT_DIR, SEG_CKPT_DIR, CLS_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

class Config:
    MODEL_NAME      = 'FAFuse-Pro'
    IMG_SIZE        = 224   # T4 15GB: 256 causes OOM with CSAF at s8/s16
    NUM_CLASSES_SEG = 1
    NUM_CLASSES_CLS = 7

    SEG_BATCH_SIZE  = 8    # T4 safe with IMG_SIZE=224 + DeiT-Small
    SEG_EPOCHS      = 60
    SEG_LR          = 1e-4
    SEG_WEIGHT_DECAY= 1e-4

    CLS_BATCH_SIZE  = 8    # physical; effective=32 via accumulation
    CLS_EPOCHS      = 60
    CLS_LR          = 2e-4
    CLS_WEIGHT_DECAY= 1e-4

    GRAD_CLIP       = 1.0
    ACCUMULATION_STEPS = 4  # effective CLS batch = 8*4 = 32

    SEG_ES_PATIENCE = 10
    SEG_ES_MIN_EPOCH= 20
    CLS_ES_PATIENCE = 10
    CLS_ES_MIN_EPOCH= 20

    SAVE_EVERY      = 5
    CLASS_NAMES     = ['MEL','NV','BCC','AKIEC','BKL','DF','VASC']
    CLS_COUNTS      = [1113, 6705, 514, 327, 1099, 115, 142]

    DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
    NUM_WORKERS     = 4
    PIN_MEMORY      = True
    SEED            = 42

cfg = Config()

# ── Speed optimizations (no quality loss) ──────────────────────
torch.backends.cudnn.benchmark        = True  # faster convs for fixed input size
torch.backends.cuda.matmul.allow_tf32 = True  # free speedup on Ampere+ GPUs
torch.backends.cudnn.allow_tf32        = True
# ───────────────────────────────────────────────────────────────

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print(f'Model   : {cfg.MODEL_NAME}')
print(f'Device  : {cfg.DEVICE}')
print(f'Output  : {OUTPUT_DIR}')

Model   : FAFuse-Pro
Device  : cuda
Output  : /kaggle/working/fafuse_pro


In [4]:
import os, glob

CLS_ROOT = "/kaggle/input/datasets/tushargaur03/isic2018/isic2018/isic2018/Task3"

for split_label, s in [('train', 'Training'), ('val', 'Validation')]:
    
    sub = os.path.join(CLS_ROOT, f'ISIC2018_Task3_{s}_GroundTruth')
    flat = os.path.join(CLS_ROOT, f'ISIC2018_Task3_{s}_GroundTruth.csv')

    print(f"\n── {split_label} ──")
    print(f"  Subdir exists   : {os.path.isdir(sub)}")

    if os.path.isdir(sub):
        contents = os.listdir(sub)
        print(f"  Subdir contents : {contents}")

        csvs = glob.glob(os.path.join(sub, '*.csv'))
        print(f"  CSVs inside     : {csvs}")

    print(f"  Flat CSV exists : {os.path.exists(flat)} → {flat}")


# Also dump top-level Task3 contents
print("\n── CLS_ROOT top-level ──")
print(os.listdir(CLS_ROOT))


── train ──
  Subdir exists   : True
  Subdir contents : ['LICENSE.txt', 'ATTRIBUTION.txt']
  CSVs inside     : []
  Flat CSV exists : True → /kaggle/input/datasets/tushargaur03/isic2018/isic2018/isic2018/Task3/ISIC2018_Task3_Training_GroundTruth.csv

── val ──
  Subdir exists   : True
  Subdir contents : ['LICENSE.txt', 'ATTRIBUTION.txt']
  CSVs inside     : []
  Flat CSV exists : True → /kaggle/input/datasets/tushargaur03/isic2018/isic2018/isic2018/Task3/ISIC2018_Task3_Validation_GroundTruth.csv

── CLS_ROOT top-level ──
['ISIC2018_Task3_Validation_GroundTruth', 'ISIC2018_Task3_Training_Input', 'ISIC2018_Task3_Validation_Input', 'ISIC2018_Task3_Training_GroundTruth', 'ISIC2018_Task3_Test_GroundTruth.csv', 'ISIC2018_Task3_Validation_GroundTruth.csv', 'ISIC2018_Task3_Test_Input', 'ISIC2018_Task3_Training_GroundTruth.csv']


In [5]:
# ── Segmentation Augmentations ────────────────────────────
def get_seg_train_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.4),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ], is_check_shapes=False)

def get_seg_val_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ], is_check_shapes=False)

# ── Classification Augmentations ──────────────────────────
cls_train_tf = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10)),
])
cls_val_tf = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Segmentation Dataset ──────────────────────────────────
class SegDataset(Dataset):
    def __init__(self, root, split='train', transform=None):
        self.transform = transform
        dirs = {
            'train': ('ISIC2018_Task1-2_Training_Input',  'ISIC2018_Task1_Training_GroundTruth'),
            'val':   ('ISIC2018_Task1-2_Validation_Input','ISIC2018_Task1_Validation_GroundTruth'),
            'test':  ('ISIC2018_Task1-2_Test_Input',      'ISIC2018_Task1_Test_GroundTruth'),
        }
        img_dir, msk_dir = [os.path.join(root, d) for d in dirs[split]]
        self.images = sorted(glob.glob(os.path.join(img_dir,'*.jpg')))
        self.masks  = sorted(glob.glob(os.path.join(msk_dir,'*.png')))
        if len(self.images) != len(self.masks):
            img_s = {os.path.splitext(os.path.basename(p))[0]: p for p in self.images}
            msk_s = {os.path.splitext(os.path.basename(p))[0].replace('_segmentation',''): p
                     for p in self.masks}
            common = sorted(set(img_s) & set(msk_s))
            self.images = [img_s[k] for k in common]
            self.masks  = [msk_s[k] for k in common]
        print(f'[Seg {split}] {len(self.images)} samples')

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert('RGB')
        mask  = Image.open(self.masks[idx]).convert('L')
        if image.size != mask.size:
            mask = mask.resize(image.size, Image.NEAREST)
        img = np.array(image)
        msk = (np.array(mask) > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=msk)
            return aug['image'], aug['mask'].unsqueeze(0)
        return torch.from_numpy(img).permute(2,0,1).float()/255, torch.from_numpy(msk).unsqueeze(0)

# ── Classification Dataset ────────────────────────────────
class ClsDataset(Dataset):
    def __init__(self, root, split='train', transform=None):
        self.transform = transform
        self.class_names = cfg.CLASS_NAMES
        split_map = {'train':'Training','val':'Validation','test':'Test'}
        s = split_map[split]
        self.img_dir = os.path.join(root, f'ISIC2018_Task3_{s}_Input')

        # ── CSV path: always flat in CLS_ROOT, subdir only has LICENSE/ATTRIBUTION ──
        if split == 'test':
            csv_path = TEST_CSV_OVERRIDE
        else:
            csv_path = os.path.join(root, f'ISIC2018_Task3_{s}_GroundTruth.csv')

        if not os.path.exists(csv_path):
            raise FileNotFoundError(
                f"[ClsDataset] Ground truth CSV not found for split='{split}'.\n"
                f"  Expected: {csv_path}"
            )

        df = pd.read_csv(csv_path)
        cols = [c for c in cfg.CLASS_NAMES if c in df.columns]
        self.labels = df[cols].values.argmax(axis=1)
        self.image_ids = df.iloc[:,0].values
        print(f'[Cls {split}] {len(self.image_ids)} samples')

    def __len__(self): return len(self.image_ids)

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, f'{self.image_ids[idx]}.jpg')
        img  = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

# ── Build Datasets ────────────────────────────────────────
print('Loading datasets...')
seg_train_ds = SegDataset(SEG_ROOT, 'train', get_seg_train_transforms(cfg.IMG_SIZE))
seg_val_ds   = SegDataset(SEG_ROOT, 'val',   get_seg_val_transforms(cfg.IMG_SIZE))
seg_test_ds  = SegDataset(SEG_ROOT, 'test',  get_seg_val_transforms(cfg.IMG_SIZE))
cls_train_ds = ClsDataset(CLS_ROOT, 'train', cls_train_tf)
cls_val_ds   = ClsDataset(CLS_ROOT, 'val',   cls_val_tf)
cls_test_ds  = ClsDataset(CLS_ROOT, 'test',  cls_val_tf)

NW = cfg.NUM_WORKERS
seg_train_dl = DataLoader(seg_train_ds, cfg.SEG_BATCH_SIZE, shuffle=True,
                          drop_last=True,
                          num_workers=NW, pin_memory=cfg.PIN_MEMORY,
                          persistent_workers=True)
seg_val_dl   = DataLoader(seg_val_ds,   cfg.SEG_BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=cfg.PIN_MEMORY,
                          persistent_workers=True)
seg_test_dl  = DataLoader(seg_test_ds,  cfg.SEG_BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=cfg.PIN_MEMORY,
                          persistent_workers=True)
cls_train_dl = DataLoader(cls_train_ds, cfg.CLS_BATCH_SIZE, shuffle=True,
                          num_workers=NW, pin_memory=cfg.PIN_MEMORY, drop_last=True,
                          persistent_workers=True)
cls_val_dl   = DataLoader(cls_val_ds,   cfg.CLS_BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=cfg.PIN_MEMORY,
                          persistent_workers=True)
cls_test_dl  = DataLoader(cls_test_ds,  cfg.CLS_BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=cfg.PIN_MEMORY,
                          persistent_workers=True)
print('DataLoaders ready.')

Loading datasets...
[Seg train] 2594 samples
[Seg val] 100 samples
[Seg test] 1000 samples
[Cls train] 10015 samples
[Cls val] 193 samples
[Cls test] 1512 samples
DataLoaders ready.


In [6]:
# ═══════════════════════════════════════════════════════════
# FAFuse-Pro — ResNet-34 + DeiT-Small + ASG + CSAF + Seg-Guided Mask
# Key idea: joint multitask model — seg mask guides classification
#   ASG  : suppresses hair/ruler artifacts at shallow CNN + deep trans
#   CSAF : asymmetric cross-attention (CNN->Q, DeiT->K/V) + SE recalib
#   mask_refine: predicted seg mask steers the cls feature (f3)
#   deep supervision: aux heads at d4, d3
# ═══════════════════════════════════════════════════════════

class ASG(nn.Module):
    # Suppresses hair/ruler artifacts using shallow CNN edges + deep Transformer context
    def __init__(self, cnn_ch, trans_ch, out_ch):
        super().__init__()
        self.cnn_proj   = nn.Conv2d(cnn_ch,    out_ch, 1)
        self.trans_proj = nn.Conv2d(trans_ch,   out_ch, 1)
        self.gate       = nn.Sequential(
            nn.Conv2d(out_ch*2, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.Sigmoid())
        self.out_proj   = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, cnn_feat, trans_feat):
        if trans_feat.shape[2:] != cnn_feat.shape[2:]:
            trans_feat = F.interpolate(trans_feat, size=cnn_feat.shape[2:],
                                       mode='bilinear', align_corners=False)
        c = self.cnn_proj(cnn_feat)
        t = self.trans_proj(trans_feat)
        g = self.gate(torch.cat([c, t], dim=1))
        return self.out_proj(t * g)

class CSAF(nn.Module):
    # Asymmetric cross-attention: CNN->Q, DeiT->K/V + SE recalibration
    def __init__(self, cnn_ch, trans_ch, out_ch, num_heads=4):
        super().__init__()
        self.q_proj = nn.Conv2d(cnn_ch,   out_ch, 1)
        self.k_proj = nn.Conv2d(trans_ch, out_ch, 1)
        self.v_proj = nn.Conv2d(trans_ch, out_ch, 1)
        self.heads  = num_heads
        self.scale  = (out_ch // num_heads) ** -0.5
        self.se_fc1 = nn.Linear(out_ch, out_ch // 4)
        self.se_fc2 = nn.Linear(out_ch // 4, out_ch)
        self.out    = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, cnn_feat, trans_feat):
        if trans_feat.shape[2:] != cnn_feat.shape[2:]:
            trans_feat = F.interpolate(trans_feat, size=cnn_feat.shape[2:],
                                       mode='bilinear', align_corners=False)
        B, C, H, W = cnn_feat.shape
        # Downsample spatial dims if too large to prevent OOM (T4 15GB guard)
        if H * W > 784:  # > 28x28: pool to 28x28 before attention
            cnn_feat   = F.adaptive_avg_pool2d(cnn_feat,   28)
            trans_feat = F.adaptive_avg_pool2d(trans_feat, 28)
            B, C, H, W = cnn_feat.shape
        Oc = self.q_proj(cnn_feat).shape[1]
        q = self.q_proj(cnn_feat).reshape(B, self.heads, Oc//self.heads, H*W).permute(0,1,3,2)
        k = self.k_proj(trans_feat).reshape(B, self.heads, Oc//self.heads, H*W).permute(0,1,3,2)
        v = self.v_proj(trans_feat).reshape(B, self.heads, Oc//self.heads, H*W).permute(0,1,3,2)
        attn = ((q @ k.transpose(-2,-1)) * self.scale).softmax(dim=-1)
        out  = (attn @ v).permute(0,1,3,2).reshape(B, Oc, H, W)
        se   = out.mean([-2,-1])
        se   = torch.sigmoid(self.se_fc2(F.relu(self.se_fc1(se)))).unsqueeze(-1).unsqueeze(-1)
        return self.out(out * se)

class LightweightGate(nn.Module):
    def __init__(self, cnn_ch, trans_ch, out_ch):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv2d(cnn_ch+trans_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, cnn_feat, trans_feat):
        if trans_feat.shape[2:] != cnn_feat.shape[2:]:
            trans_feat = F.interpolate(trans_feat, size=cnn_feat.shape[2:],
                                       mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([cnn_feat, trans_feat], dim=1))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch//2, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch//2 + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class FAFusePro(nn.Module):
    def __init__(self, num_seg=1, num_cls=7):
        super().__init__()
        # CNN backbone: ResNet-34 (ImageNet pretrained)
        self.cnn      = timm.create_model('resnet34', pretrained=True, features_only=True)
        cnn_chs       = [64, 64, 128, 256, 512]

        # Transformer: DeiT-Small (pretrained, 256x256 input)
        self.trans     = timm.create_model('deit_small_patch16_224', pretrained=True,
                                           img_size=cfg.IMG_SIZE, dynamic_img_size=True, num_classes=0)
        self.trans_dim = self.trans.embed_dim   # 384
        nblocks        = len(self.trans.blocks) # 12
        self.splits    = [nblocks//4, nblocks//2, 3*nblocks//4, nblocks]
        trans_out_chs  = [64, 128, 256, 512]
        self.trans_proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(self.trans_dim, ch, 1),
                          nn.BatchNorm2d(ch), nn.ReLU(inplace=True))
            for ch in trans_out_chs])

        # ASG: shallow CNN (c[0]) + deepest trans token -> artifact suppression
        self.asg = ASG(cnn_chs[0], trans_out_chs[3], trans_out_chs[3])

        # Fusion: LightweightGate at s4, CSAF at s8/s16/s32
        self.gate_s4  = LightweightGate(cnn_chs[1], trans_out_chs[0], 64)
        self.csaf_s8  = CSAF(cnn_chs[2], trans_out_chs[1], 128)
        self.csaf_s16 = CSAF(cnn_chs[3], trans_out_chs[2], 256)
        self.csaf_s32 = CSAF(cnn_chs[4], trans_out_chs[3], 512)

        # Decoder
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128,  64,  64)
        self.dec1 = DecoderBlock( 64,  64,  32)

        # Deep supervision auxiliary heads
        self.aux_d3 = nn.Conv2d(256, num_seg, 1)
        self.aux_d2 = nn.Conv2d(128, num_seg, 1)

        # Segmentation head
        self.seg_head = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 2, stride=2),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, num_seg, 1))

        # Seg-guided mask refine: couples seg output into cls feature
        self.mask_refine = nn.Sequential(
            nn.Conv2d(num_seg, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(7), nn.Conv2d(32, 1, 1), nn.Sigmoid())

        # Classification head (seg-guided: uses f3 attentionally modulated by seg mask)
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(256, num_cls))

    def _trans_features(self, x):
        x   = self.trans.patch_embed(x)
        if x.ndim == 4:          # dynamic_img_size=True returns (B, H, W, C)
            B, H, W, C = x.shape
            x = x.reshape(B, H * W, C)
        cls = self.trans.cls_token.expand(x.shape[0], -1, -1)
        x   = torch.cat([cls, x], dim=1) + self.trans.pos_embed
        x   = self.trans.pos_drop(x)
        feats, prev = [], 0
        for lvl, sp in enumerate(self.splits):
            for blk in self.trans.blocks[prev:sp]: x = blk(x)
            f = x[:,1:,:]; B,N,D = f.shape; H=W=int(math.sqrt(N))
            feats.append(self.trans_proj[lvl](f.transpose(1,2).reshape(B,D,H,W)))
            prev = sp
        return feats   # [s4, s8, s16, s32]

    def forward(self, x, task='seg', warmup=False):
        c = self.cnn(x)          # c[0]=stem, c[1..4]=stages
        t = self._trans_features(x)

        # ASG on deepest transformer output using shallow CNN edges
        t3_clean = self.asg(c[0], t[3])

        # Fuse CNN + Transformer at each scale
        f0 = self.gate_s4(c[1], t[0])       # 56x56 (or 64x64 at 256 input)
        f1 = self.csaf_s8(c[2],  t[1])      # 28x28
        f2 = self.csaf_s16(c[3], t[2])      # 14x14
        f3 = self.csaf_s32(c[4], t3_clean)  # 7x7 (artifact-suppressed)

        # Decoder with skip connections
        d4 = self.dec4(f3, f2)   # 14x14, 256ch
        d3 = self.dec3(d4, f1)   # 28x28, 128ch
        d2 = self.dec2(d3, f0)   # 56x56,  64ch
        d1 = self.dec1(d2, c[0]) # 112x112, 32ch

        seg = self.seg_head(d1)
        if seg.shape[2:] != x.shape[2:]:
            seg = F.interpolate(seg, size=x.shape[2:], mode='bilinear', align_corners=False)

        if task == 'seg': return seg

        # Seg-guided feature modulation for classification
        # During warmup seg mask is unreliable — use unmodulated f3
        if warmup:
            cls_feat = f3
        else:
            mask_s   = self.mask_refine(seg.detach())   # detach: no seg grad from cls path
            mask_up  = F.interpolate(mask_s, size=f3.shape[2:], mode='bilinear', align_corners=False)
            cls_feat = f3 * (1 + mask_up)               # soft attention boost on lesion region

        cls_out = self.cls_head(cls_feat)

        if task == 'cls': return cls_out

        # task == 'both': joint training with deep supervision
        aux3 = F.interpolate(self.aux_d3(d4), size=x.shape[2:], mode='bilinear', align_corners=False)
        aux2 = F.interpolate(self.aux_d2(d3), size=x.shape[2:], mode='bilinear', align_corners=False)
        return seg, cls_out, aux3, aux2

model = FAFusePro().to(cfg.DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'FAFuse-Pro | Params: {total:,}')
x_t = torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).to(cfg.DEVICE)
with torch.no_grad():
    s = model(x_t, task='seg')
    c2 = model(x_t, task='cls')
    seg2, cls2, a3, a2 = model(x_t, task='both', warmup=False)
print(f'Seg: {s.shape} | Cls: {c2.shape} | Aux3: {a3.shape} | Aux2: {a2.shape}')
del x_t; torch.cuda.empty_cache()
print('✓ FAFuse-Pro sanity check passed')


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

FAFuse-Pro | Params: 53,359,723
Seg: torch.Size([2, 1, 224, 224]) | Cls: torch.Size([2, 7]) | Aux3: torch.Size([2, 1, 224, 224]) | Aux2: torch.Size([2, 1, 224, 224])
✓ FAFuse-Pro sanity check passed


In [7]:
# ── Segmentation Loss: Dice + BCE (NaN guard) ─────────────────────────

def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred)
    flat_p = pred.reshape(-1)
    flat_t = target.reshape(-1)

    inter = (flat_p * flat_t).sum()

    return 1 - (2 * inter + smooth) / (flat_p.sum() + flat_t.sum() + smooth)


def seg_loss_fn(pred, target):
    dl = dice_loss(pred, target)
    bl = F.binary_cross_entropy_with_logits(pred, target)

    loss = dl + bl

    if not torch.isfinite(loss):
        return torch.tensor(0.0, requires_grad=True, device=pred.device)

    return loss


# ── Focal Loss (for classification) ───────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ls = label_smoothing

    def forward(self, logits, targets):
        n_cls = logits.size(1)

        with torch.no_grad():
            soft = torch.full_like(logits, self.ls / (n_cls - 1))
            soft.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)

        log_p = F.log_softmax(logits, dim=1)
        ce = -(soft * log_p).sum(dim=1)

        p_t = torch.exp(-ce)
        focal = ((1 - p_t) ** self.gamma) * ce

        if self.weight is not None:
            focal = focal * self.weight[targets]

        return focal.mean()


# ── Class Weights ─────────────────────────────────────────────────────

_counts = cfg.CLS_COUNTS
_total = sum(_counts)

_cls_weights = torch.tensor(
    [math.sqrt(_total / (cfg.NUM_CLASSES_CLS * c)) for c in _counts],
    dtype=torch.float
).to(cfg.DEVICE)

cls_criterion = FocalLoss(
    gamma=2.0,
    weight=_cls_weights,
    label_smoothing=0.05
)

print('Class weights:', [f'{w:.3f}' for w in _cls_weights.cpu().tolist()])


# ── Dice Score (metric) ───────────────────────────────────────────────

def dice_score(pred, target, thr=0.5):
    pred_b = (torch.sigmoid(pred) > thr).float()
    inter = (pred_b * target).sum()

    return ((2 * inter + 1) / (pred_b.sum() + target.sum() + 1)).item()


print('✓ Loss functions ready')

Class weights: ['1.134', '0.462', '1.668', '2.092', '1.141', '3.527', '3.174']
✓ Loss functions ready


In [8]:
# ── Validation helpers ───────────────────────────────────────────────

@torch.no_grad()
def val_seg(loader, mdl=None):
    if mdl is None:
        mdl = model

    mdl.eval()
    tp = fp = fn = tn = 0.0

    for imgs, msks in loader:
        imgs = imgs.to(cfg.DEVICE, non_blocking=True)
        msks = msks.to(cfg.DEVICE, non_blocking=True)

        out = mdl(imgs, task='seg')
        pred = out[0] if isinstance(out, tuple) else out

        pb = (torch.sigmoid(pred) > 0.5).float()
        mb = (msks > 0.5).float()

        tp += (pb * mb).sum().item()
        fp += (pb * (1 - mb)).sum().item()
        fn += ((1 - pb) * mb).sum().item()
        tn += ((1 - pb) * (1 - mb)).sum().item()

    eps = 1e-7

    return {
        'dice': 2 * tp / (2 * tp + fp + fn + eps),
        'iou': tp / (tp + fp + fn + eps),
        'recall': tp / (tp + fn + eps),
        'precision': tp / (tp + fp + eps),
        'pa': (tp + tn) / (tp + tn + fp + fn + eps)
    }


@torch.no_grad()
def val_cls(loader, mdl, task_key='cls'):
    mdl.eval()
    all_labels, all_probs = [], []

    for imgs, labels in loader:
        imgs = imgs.to(cfg.DEVICE, non_blocking=True)

        logits = mdl(imgs, task=task_key)

        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

    probs = np.vstack(all_probs)
    labels = np.array(all_labels)

    preds = probs.argmax(1)

    acc = accuracy_score(labels, preds)

    try:
        auc = roc_auc_score(labels, probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    f1 = f1_score(labels, preds, average='weighted', zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)

    return {
        'acc': acc,
        'auc': auc,
        'f1': f1,
        'f1_macro': f1_macro,
        'labels': labels,
        'probs': probs,
        'preds': preds
    }


print('✓ Val helpers ready')

✓ Val helpers ready


In [9]:
# ── WeightedRandomSampler + MixUp ───────────────────────────
from torch.utils.data import WeightedRandomSampler

_sample_w = [1.0 / cfg.CLS_COUNTS[lbl] for lbl in cls_train_ds.labels]
_sampler  = WeightedRandomSampler(
    weights=torch.tensor(_sample_w, dtype=torch.float),
    num_samples=len(_sample_w),
    replacement=True)

cls_train_dl = DataLoader(
    cls_train_ds, batch_size=cfg.CLS_BATCH_SIZE,
    sampler=_sampler, drop_last=True,
    num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
    persistent_workers=cfg.NUM_WORKERS > 0,
    prefetch_factor=2 if cfg.NUM_WORKERS > 0 else None)

def mixup_batch(imgs, labels, alpha=0.2, num_classes=7):
    lam   = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    bs    = imgs.size(0)
    idx   = torch.randperm(bs, device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    y_a   = F.one_hot(labels, num_classes).float()
    y_b   = F.one_hot(labels[idx], num_classes).float()
    soft  = lam * y_a + (1 - lam) * y_b
    return mixed, soft

print(f'✓ WeightedRandomSampler ready | {len(_sample_w)} samples')
print('✓ MixUp helper ready (alpha=0.2)')

✓ WeightedRandomSampler ready | 10015 samples
✓ MixUp helper ready (alpha=0.2)


## FAFuse-Pro Training (v2: Separate Seg → Cls Phases)

In [10]:
# ── VRAM check before training ──────────────────────────────
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**2
    alloc_vram = torch.cuda.memory_allocated() / 1024**2
    print(f'VRAM: {alloc_vram:.0f} / {total_vram:.0f} MB allocated before seg training')
    if alloc_vram > 0.7 * total_vram:
        print('WARNING: >70% VRAM used before training — clearing cache')
        torch.cuda.empty_cache()
        import gc; gc.collect()

# ══════════════════════════════════════════════════════════════
# FAFuse-Pro SEGMENTATION TRAINING (Task 1) — separate phase
# No warmup; direct segmentation training with deep supervision
# ══════════════════════════════════════════════════════════════
def build_seg_optimizer(mdl):
    _trans_ids = {id(p) for p in mdl.trans.parameters()}
    _trans_p   = [p for p in mdl.parameters() if id(p) in _trans_ids]
    _other_p   = [p for p in mdl.parameters() if id(p) not in _trans_ids]
    print(f'  Seg optimizer: split LR — trans {cfg.SEG_LR*0.1:.1e}, rest {cfg.SEG_LR:.1e}')
    return AdamW([{'params': _trans_p, 'lr': cfg.SEG_LR*0.1},
                  {'params': _other_p, 'lr': cfg.SEG_LR}],
                 weight_decay=cfg.SEG_WEIGHT_DECAY)

seg_optimizer = build_seg_optimizer(model)
seg_scheduler = CosineAnnealingLR(seg_optimizer, T_max=cfg.SEG_EPOCHS * 2, eta_min=1e-6)
seg_scaler    = GradScaler('cuda')

start_epoch_seg = 0; best_seg_dice = 0.0; seg_es_counter = 0; seg_log = []
seg_ckpts = sorted(glob.glob(os.path.join(SEG_CKPT_DIR, 'epoch_*.pt')))
if seg_ckpts:
    ck = torch.load(seg_ckpts[-1], map_location=cfg.DEVICE, weights_only=False)
    model.load_state_dict(ck['model'])
    seg_optimizer.load_state_dict(ck['optimizer'])
    seg_scheduler.load_state_dict(ck['scheduler'])
    start_epoch_seg = ck['epoch'] + 1
    best_seg_dice   = ck['best_dice']
    seg_es_counter  = ck.get('seg_es_counter', 0)
    if os.path.exists(SEG_LOG_PATH):
        with open(SEG_LOG_PATH) as f: seg_log = json.load(f)
    print(f'✓ Seg resumed from epoch {start_epoch_seg} | best_dice={best_seg_dice:.4f} | ES={seg_es_counter}/{cfg.SEG_ES_PATIENCE}')
else:
    print('Starting fresh seg training')

DEEP_SUP_W3 = 0.4
DEEP_SUP_W2 = 0.2

t0_total = time.time()

for epoch in range(start_epoch_seg, cfg.SEG_EPOCHS):
    model.train()
    loss_sum, n = 0.0, 0
    t0 = time.time()

    for imgs, msks in tqdm(seg_train_dl, desc=f'Seg Ep{epoch+1:03d}', leave=False):
        imgs, msks = imgs.to(cfg.DEVICE, non_blocking=True), msks.to(cfg.DEVICE, non_blocking=True)
        seg_optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            out = model(imgs, task='seg')
            if isinstance(out, tuple):
                pred_seg, _, aux3, aux2 = out
                loss = (seg_loss_fn(pred_seg, msks)
                        + DEEP_SUP_W3 * seg_loss_fn(aux3, msks)
                        + DEEP_SUP_W2 * seg_loss_fn(aux2, msks))
            else:
                loss = seg_loss_fn(out, msks)
        if not torch.isfinite(loss):
            seg_scaler.update(); continue
        seg_scaler.scale(loss).backward()
        seg_scaler.unscale_(seg_optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        seg_scaler.step(seg_optimizer)
        seg_scaler.update()
        loss_sum += loss.item(); n += 1

    seg_scheduler.step()
    m = val_seg(seg_val_dl, model)
    is_best = m['dice'] > best_seg_dice
    if is_best:
        best_seg_dice = m['dice']; seg_es_counter = 0
    else:
        seg_es_counter += 1

    ep_time = time.time() - t0
    elapsed = time.time() - t0_total
    eta     = elapsed / (epoch - start_epoch_seg + 1) * (cfg.SEG_EPOCHS - epoch - 1)
    es_tag  = f'ES={seg_es_counter}/{cfg.SEG_ES_PATIENCE}'
    print(f"Ep {epoch+1:03d}/{cfg.SEG_EPOCHS} | "
          f"Loss={loss_sum/max(n,1):.4f} | Dice={m['dice']:.4f} | IoU={m['iou']:.4f} | "
          f"Rec={m['recall']:.4f} | Prec={m['precision']:.4f} | "
          f"{ep_time/60:.1f}min | ETA={eta/60:.0f}min | {es_tag} {'★' if is_best else ''}")

    entry = {'epoch': epoch+1, 'train_loss': round(loss_sum/max(n,1),4),
             'dice': round(m['dice'],4), 'iou': round(m['iou'],4),
             'recall': round(m['recall'],4), 'precision': round(m['precision'],4), 'pa': round(m['pa'],4)}
    seg_log.append(entry)
    with open(SEG_LOG_PATH, 'w') as f: json.dump(seg_log, f, indent=2)

    ck = {'epoch': epoch, 'model': model.state_dict(),
          'optimizer': seg_optimizer.state_dict(),
          'scheduler': seg_scheduler.state_dict(),
          'best_dice': best_seg_dice, 'seg_es_counter': seg_es_counter}
    if is_best or (epoch+1) % cfg.SAVE_EVERY == 0:
        torch.save(ck, os.path.join(SEG_CKPT_DIR, f'epoch_{epoch:03d}.pt'))
    if is_best:
        torch.save(ck, os.path.join(SEG_CKPT_DIR, 'best_seg.pt'))

    for old in sorted(glob.glob(os.path.join(SEG_CKPT_DIR,'epoch_*.pt')))[:-3]:
        os.remove(old)

    if epoch + 1 >= cfg.SEG_ES_MIN_EPOCH and seg_es_counter >= cfg.SEG_ES_PATIENCE:
        print(f'⚡ Seg early stopping triggered at epoch {epoch+1}')
        break

print(f'\n✓ Seg training done | Best Val Dice: {best_seg_dice:.4f}')

# ── Memory cleanup after seg training ────────────────────────
# Free seg optimizer/scaler before cls phase to reclaim ~500MB VRAM
del seg_optimizer, seg_scaler
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'VRAM after seg cleanup: {torch.cuda.memory_allocated()/1024**2:.0f} MB allocated')


VRAM: 218 / 14913 MB allocated before seg training
  Seg optimizer: split LR — trans 1.0e-05, rest 1.0e-04
Starting fresh seg training


Seg Ep001:   0%|          | 0/324 [00:00<?, ?it/s]

Ep 001/60 | Loss=0.9644 | Dice=0.7934 | IoU=0.6575 | Rec=0.9864 | Prec=0.6636 | 5.0min | ETA=294min | ES=0/10 ★


Seg Ep002:   0%|          | 0/324 [00:00<?, ?it/s]

Ep 002/60 | Loss=0.7453 | Dice=0.8360 | IoU=0.7183 | Rec=0.9753 | Prec=0.7316 | 4.2min | ETA=268min | ES=0/10 ★


...

Ep 034/60 | Loss=0.1026 | Dice=0.8978 | IoU=0.8146 | Rec=0.9698 | Prec=0.8358 | 4.2min | ETA=110min | ES=8/10 


Seg Ep035:   0%|          | 0/324 [00:00<?, ?it/s]

Ep 035/60 | Loss=0.1013 | Dice=0.8993 | IoU=0.8170 | Rec=0.9516 | Prec=0.8525 | 4.2min | ETA=106min | ES=9/10 


Seg Ep036:   0%|          | 0/324 [00:00<?, ?it/s]

Ep 036/60 | Loss=0.0993 | Dice=0.9011 | IoU=0.8201 | Rec=0.9428 | Prec=0.8630 | 4.3min | ETA=102min | ES=10/10 
⚡ Seg early stopping triggered at epoch 36

✓ Seg training done | Best Val Dice: 0.9029
VRAM after seg cleanup: 856 MB allocated


In [11]:
# ── Load best seg model & evaluate on test set ───────────────────────

best_ckpt = os.path.join(SEG_CKPT_DIR, 'best_seg.pt')

ck = torch.load(
    best_ckpt,
    map_location=cfg.DEVICE,
    weights_only=False
)

model.load_state_dict(ck['model'])

print(f"Loaded best seg model (epoch {ck['epoch']+1}, val dice={ck['best_dice']:.4f})")

model.eval()

tp = fp = fn = tn = 0.0

with torch.no_grad():
    for imgs, msks in tqdm(seg_test_dl, desc='Seg test'):

        imgs = imgs.to(cfg.DEVICE, non_blocking=True)
        msks = msks.to(cfg.DEVICE, non_blocking=True)

        out = model(imgs, task='seg')
        pred = out[0] if isinstance(out, tuple) else out

        pb = (torch.sigmoid(pred) > 0.5).float()
        mb = (msks > 0.5).float()

        tp += (pb * mb).sum().item()
        fp += (pb * (1 - mb)).sum().item()
        fn += ((1 - pb) * mb).sum().item()
        tn += ((1 - pb) * (1 - mb)).sum().item()

eps = 1e-7

seg_results = {
    'Dice': 2 * tp / (2 * tp + fp + fn + eps),
    'IoU': tp / (tp + fp + fn + eps),
    'Recall': tp / (tp + fn + eps),
    'Precision': tp / (tp + fp + eps),
    'PA': (tp + tn) / (tp + tn + fp + fn + eps),
}

print("\n── Segmentation (Task 1) Test Results ──")
for k, v in seg_results.items():
    print(f"{k:12s}: {v:.4f}")

Loaded best seg model (epoch 26, val dice=0.9029)


Seg test:   0%|          | 0/125 [00:00<?, ?it/s]


── Segmentation (Task 1) Test Results ──
Dice        : 0.8819
IoU         : 0.7887
Recall      : 0.9210
Precision   : 0.8460
PA          : 0.9310


## FAFuse-Pro Classification Training (Task 3) — separate phase

In [12]:
# ── FAFuse-Pro Classification Training (Task 3) ──────────────────────

model_cls = FAFusePro().to(cfg.DEVICE)  # fresh model


def build_cls_optimizer(m):
    trans_ids = {id(p) for p in m.trans.parameters()}

    trans_p = [p for p in m.parameters() if id(p) in trans_ids]
    other_p = [p for p in m.parameters() if id(p) not in trans_ids]

    print(f'Cls optimizer: split LR — trans {cfg.CLS_LR*0.1:.1e}, rest {cfg.CLS_LR:.1e}')

    return AdamW(
        [
            {'params': trans_p, 'lr': cfg.CLS_LR * 0.1},
            {'params': other_p, 'lr': cfg.CLS_LR}
        ],
        weight_decay=cfg.CLS_WEIGHT_DECAY
    )


cls_optimizer = build_cls_optimizer(model_cls)
cls_sched = StepLR(cls_optimizer, step_size=15, gamma=0.3)
cls_scaler = GradScaler('cuda')


start_epoch_cls = 0
best_cls_score = 0.0
cls_es_counter = 0
cls_log = []

# ── Resume from checkpoint if available ──────────────────────────────
cls_ckpts = sorted(glob.glob(os.path.join(CLS_CKPT_DIR, 'epoch_*.pt')))
if cls_ckpts:
    ck = torch.load(cls_ckpts[-1], map_location=cfg.DEVICE, weights_only=False)
    model_cls.load_state_dict(ck['model'])
    cls_optimizer.load_state_dict(ck['optimizer'])
    cls_sched.load_state_dict(ck['scheduler'])
    start_epoch_cls = ck['epoch'] + 1
    best_cls_score  = ck['best_cls_score']
    cls_es_counter  = ck.get('cls_es_counter', 0)
    if os.path.exists(CLS_LOG_PATH):
        with open(CLS_LOG_PATH) as f: cls_log = json.load(f)
    print(f'✓ Cls resumed from epoch {start_epoch_cls} | best_score={best_cls_score:.4f} | ES={cls_es_counter}/{cfg.CLS_ES_PATIENCE}')
else:
    # Warm-start cls backbone from best seg checkpoint
    seg_best = os.path.join(SEG_CKPT_DIR, 'best_seg.pt')
    if os.path.exists(seg_best):
        ck = torch.load(seg_best, map_location=cfg.DEVICE, weights_only=False)
        model_cls.load_state_dict(ck['model'])
        print(f'✓ Cls warm-started from best seg ckpt (dice={ck["best_dice"]:.4f})')
    else:
        print('Starting fresh cls training (no seg ckpt found)')

t0_total = time.time()

for epoch in range(start_epoch_cls, cfg.CLS_EPOCHS):
    model_cls.train()
    loss_sum, n = 0.0, 0
    t0 = time.time()

    warmup = epoch < 5  # first 5 epochs: seg mask unreliable, skip mask refine

    for imgs, labels in tqdm(cls_train_dl, desc=f'Cls Ep{epoch+1:03d}', leave=False):
        imgs   = imgs.to(cfg.DEVICE, non_blocking=True)
        labels = labels.to(cfg.DEVICE, non_blocking=True)

        cls_optimizer.zero_grad(set_to_none=True)

        # MixUp
        imgs_m, soft_labels = mixup_batch(imgs, labels, alpha=0.2,
                                          num_classes=cfg.NUM_CLASSES_CLS)

        with autocast('cuda'):
            logits = model_cls(imgs_m, task='cls', warmup=warmup)
            # Focal loss expects hard labels; use soft targets via manual CE
            log_p  = F.log_softmax(logits, dim=1)
            loss   = -(soft_labels * log_p).sum(dim=1).mean()

        if not torch.isfinite(loss):
            cls_scaler.update(); continue

        cls_scaler.scale(loss).backward()
        cls_scaler.unscale_(cls_optimizer)
        torch.nn.utils.clip_grad_norm_(model_cls.parameters(), cfg.GRAD_CLIP)
        cls_scaler.step(cls_optimizer)
        cls_scaler.update()

        loss_sum += loss.item(); n += 1

    cls_sched.step()

    m = val_cls(cls_val_dl, model_cls, task_key='cls')
    score = 0.5 * m['auc'] + 0.5 * m['acc']   # composite for ES/ckpt

    is_best = score > best_cls_score
    if is_best:
        best_cls_score = score; cls_es_counter = 0
    else:
        cls_es_counter += 1

    ep_time = time.time() - t0
    elapsed = time.time() - t0_total
    eta     = elapsed / (epoch - start_epoch_cls + 1) * (cfg.CLS_EPOCHS - epoch - 1)
    es_tag  = f'ES={cls_es_counter}/{cfg.CLS_ES_PATIENCE}'
    print(f"Ep {epoch+1:03d}/{cfg.CLS_EPOCHS} | "
          f"Loss={loss_sum/max(n,1):.4f} | Acc={m['acc']:.4f} | AUC={m['auc']:.4f} | "
          f"F1w={m['f1']:.4f} | F1mac={m['f1_macro']:.4f} | "
          f"{ep_time/60:.1f}min | ETA={eta/60:.0f}min | {es_tag} {'★' if is_best else ''}")

    entry = {'epoch': epoch+1, 'train_loss': round(loss_sum/max(n,1),4),
             'acc': round(m['acc'],4), 'auc': round(m['auc'],4),
             'f1': round(m['f1'],4), 'f1_macro': round(m['f1_macro'],4)}
    cls_log.append(entry)
    with open(CLS_LOG_PATH, 'w') as f: json.dump(cls_log, f, indent=2)

    ck = {'epoch': epoch, 'model': model_cls.state_dict(),
          'optimizer': cls_optimizer.state_dict(),
          'scheduler': cls_sched.state_dict(),
          'best_cls_score': best_cls_score, 'cls_es_counter': cls_es_counter}
    if is_best or (epoch+1) % cfg.SAVE_EVERY == 0:
        torch.save(ck, os.path.join(CLS_CKPT_DIR, f'epoch_{epoch:03d}.pt'))
    if is_best:
        torch.save(ck, os.path.join(CLS_CKPT_DIR, 'best_cls.pt'))

    for old in sorted(glob.glob(os.path.join(CLS_CKPT_DIR,'epoch_*.pt')))[:-3]:
        os.remove(old)

    if epoch + 1 >= cfg.CLS_ES_MIN_EPOCH and cls_es_counter >= cfg.CLS_ES_PATIENCE:
        print(f'⚡ Cls early stopping triggered at epoch {epoch+1}')
        break

print(f'\n✓ Cls training done | Best Val Score (0.5*AUC+0.5*Acc): {best_cls_score:.4f}')

# ── Cleanup ────────────────────────────────────────────────────────
del cls_optimizer, cls_scaler
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'VRAM after cls cleanup: {torch.cuda.memory_allocated()/1024**2:.0f} MB allocated')


Cls optimizer: split LR — trans 2.0e-05, rest 2.0e-04
✓ Cls warm-started from best seg ckpt (dice=0.9029)


Cls Ep001:   0%|          | 0/1251 [00:00<?, ?it/s]

Ep 001/60 | Loss=1.2838 | Acc=0.7409 | AUC=0.9397 | F1w=0.7639 | F1mac=0.5772 | 6.4min | ETA=380min | ES=0/10 ★


Cls Ep002:   0%|          | 0/1251 [00:00<?, ?it/s]

Ep 002/60 | Loss=1.0233 | Acc=0.7358 | AUC=0.9518 | F1w=0.7607 | F1mac=0.6746 | 6.4min | ETA=373min | ES=0/10 ★


...

Ep 018/60 | Loss=0.4651 | Acc=0.7979 | AUC=0.9587 | F1w=0.8093 | F1mac=0.6304 | 6.2min | ETA=266min | ES=8/10 


Cls Ep019:   0%|          | 0/1251 [00:00<?, ?it/s]

Ep 019/60 | Loss=0.4334 | Acc=0.8083 | AUC=0.9580 | F1w=0.8199 | F1mac=0.7624 | 6.2min | ETA=259min | ES=9/10 


Cls Ep020:   0%|          | 0/1251 [00:00<?, ?it/s]

Ep 020/60 | Loss=0.4423 | Acc=0.7772 | AUC=0.9519 | F1w=0.7968 | F1mac=0.7511 | 6.2min | ETA=253min | ES=10/10 
⚡ Cls early stopping triggered at epoch 20

✓ Cls training done | Best Val Score (0.5*AUC+0.5*Acc): 0.9190
VRAM after cls cleanup: 1651 MB allocated


In [13]:
# ── Load best cls model & evaluate on test set ───────────────────────

ck = torch.load(
    os.path.join(CLS_CKPT_DIR, 'best_cls.pt'),
    map_location=cfg.DEVICE,
    weights_only=False
)

model_cls.load_state_dict(ck['model'])

print(f"Loaded best cls model (epoch {ck['epoch']+1}, val score={ck['best_cls_score']:.4f})")

all_labels, all_probs = [], []

model_cls.eval()

with torch.no_grad():
    for imgs, labels in tqdm(cls_test_dl, desc='Cls test'):
        imgs = imgs.to(cfg.DEVICE, non_blocking=True)

        logits = model_cls(imgs, task='cls')

        probs_batch = torch.softmax(logits, dim=1).detach().cpu().numpy()

        all_probs.append(probs_batch)
        all_labels.extend(labels.numpy())

probs = np.vstack(all_probs)
labels = np.array(all_labels)

preds = probs.argmax(1)

# ── Metrics ─────────────────────────────────────────────────────────

acc = accuracy_score(labels, preds)

try:
    auc = roc_auc_score(labels, probs, multi_class='ovr', average='weighted')
except:
    auc = 0.0

f1_w = f1_score(labels, preds, average='weighted', zero_division=0)
prec_w = precision_score(labels, preds, average='weighted', zero_division=0)
rec_w = recall_score(labels, preds, average='weighted', zero_division=0)

cls_results = {
    'Accuracy': acc,
    'AUC (weighted)': auc,
    'F1 (weighted)': f1_w,
    'Precision': prec_w,
    'Recall': rec_w
}

print("\n── Classification (Task 3) Test Results ──")
for k, v in cls_results.items():
    print(f"{k:18s}: {v:.4f}")

print("\n── Per-class Report ──")
print(classification_report(
    labels,
    preds,
    target_names=cfg.CLASS_NAMES,
    zero_division=0
))

# ── Save results ─────────────────────────────────────────────────────

results_path = os.path.join(OUTPUT_DIR, 'test_results.json')

with open(results_path, 'w') as f:
    json.dump({
        'seg': seg_results,
        'cls': cls_results
    }, f, indent=2)

print(f"\nSaved to {results_path}")

Loaded best cls model (epoch 10, val score=0.9190)


Cls test:   0%|          | 0/189 [00:00<?, ?it/s]


── Classification (Task 3) Test Results ──
Accuracy          : 0.7976
AUC (weighted)    : 0.9438
F1 (weighted)     : 0.7980
Precision         : 0.8010
Recall            : 0.7976

── Per-class Report ──
              precision    recall  f1-score   support

         MEL       0.52      0.57      0.54       171
          NV       0.91      0.88      0.89       909
         BCC       0.64      0.74      0.69        93
       AKIEC       0.44      0.26      0.32        43
         BKL       0.71      0.75      0.73       217
          DF       0.84      0.82      0.83        44
        VASC       0.85      0.80      0.82        35

    accuracy                           0.80      1512
   macro avg       0.70      0.69      0.69      1512
weighted avg       0.80      0.80      0.80      1512


Saved to /kaggle/working/fafuse_pro/test_results.json


In [14]:
import zipfile
import os, glob

zip_path = f"/kaggle/working/{cfg.MODEL_NAME.lower().replace('-', '_')}_results.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:

    # ── Seg checkpoint ──
    for ckpt in glob.glob(os.path.join(SEG_CKPT_DIR, 'best_seg.pt')):
        zf.write(ckpt, 'seg_checkpoints/best_seg.pt')

    # ── Cls checkpoint ──
    for ckpt in glob.glob(os.path.join(CLS_CKPT_DIR, 'best_cls.pt')):
        zf.write(ckpt, 'cls_checkpoints/best_cls.pt')

    # ── JSON results ──
    for f in glob.glob(os.path.join(OUTPUT_DIR, '*.json')):
        zf.write(f, os.path.basename(f))

    # ── PNG outputs (plots, confusion matrix etc.) ──
    for f in glob.glob(os.path.join(OUTPUT_DIR, '*.png')):
        zf.write(f, os.path.basename(f))


size_mb = os.path.getsize(zip_path) / 1024**2

print(f"✓ Saved: {zip_path} ({size_mb:.1f} MB)")

✓ Saved: /kaggle/working/fafuse_pro_results.zip (1093.8 MB)


## Changes Applied (FAFuse-Pro v2)

| Change | Old | New |
|--------|-----|-----|
| Training strategy | Joint warmup+cls (1 optimizer) | **Separate seg/cls phases** (2 independent trainings) |
| `CLS_LR` | `1e-4` | `2e-4` |
| `SEG/CLS_EPOCHS` | `50` | `60` |
| `CLS_SCHEDULER` | `CosineAnnealingLR` | `StepLR(step=15, γ=0.3)` |
| Loss function | `WeightedCE (LS=0.1)` | `FocalLoss(γ=2, LS=0.05)` |
| Sampling | `shuffle=True` | `WeightedRandomSampler` |
| Augmentation | Basic | `+ MixUp(α=0.2)` |
| NaN guard | None | skip batch + reset scaler |
| Seg/Cls ES | `patience=15, min=25` | `patience=10, min=20` |
| Deep supervision | In joint pass | In seg-only pass (same weights 0.4/0.2) |